In [2]:
import time
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score,
)


In [3]:
df = pd.read_csv(r'pipeline_cache\ampds_behavior_context_labeled_features.csv')
df = df.drop(columns=['window_id']) 
print(df.shape)

(5856, 832)


In [4]:
df_reset = df.reset_index(drop=True)

anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
normal_df = df_reset[df_reset['is_anomaly'] == 0]

anom_train, anom_temp = train_test_split(
    anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
)
anom_val, anom_test = train_test_split(
    anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
)

norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

print(pd.crosstab(
    pd.concat([anom_train, anom_val, anom_test])['anomaly_type'],
    pd.concat([anom_train.assign(split='train'),
               anom_val.assign(split='val'),
               anom_test.assign(split='test')])['split']
))

split                             test  train  val
anomaly_type                                      
appliance_unusual_hours             12     54   12
gradual_drift_decrease               3     16    4
gradual_drift_increase               6     28    6
heating_on_warm_day                  9     44   10
high_usage_low_occupancy            12     54   11
impossible_appliance_combo           5     22    4
multiple_high_power_simultaneous     5     24    5
power_spike                          5     24    5
sensor_glitch                        4     17    3
stuck_appliance_off                 13     61   14
stuck_appliance_on                  14     62   13
sustained_overload                   8     37    8
weekday_pattern_on_weekend           9     43    9
weekend_pattern_on_weekday           9     44   10


C:\Users\revan\AppData\Local\Temp\ipykernel_25732\1503380893.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  pd.concat([anom_train.assign(split='train'),
C:\Users\revan\AppData\Local\Temp\ipykernel_25732\1503380893.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  anom_val.assign(split='val'),
C:\Users\revan\AppData\Local\Temp\ipykernel_25732\1503380893.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining

In [5]:
drop_cols = ['is_anomaly', 'anomaly_type']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_anomaly']

X_val = val_df.drop(columns=drop_cols)
y_val = val_df['is_anomaly']

X_test = test_df.drop(columns=drop_cols)
y_test = test_df['is_anomaly']

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

print("\nTrain anomaly type counts:\n", train_df['anomaly_type'].value_counts())
print("\nVal anomaly type counts:\n", val_df['anomaly_type'].value_counts())
print("\nTest anomaly type counts:\n", test_df['anomaly_type'].value_counts())


(4098, 830) (4098,)
(879, 830) (879,)
(879, 830) (879,)

Train anomaly type counts:
 anomaly_type
normal                              3568
stuck_appliance_on                    62
stuck_appliance_off                   61
high_usage_low_occupancy              54
appliance_unusual_hours               54
heating_on_warm_day                   44
weekend_pattern_on_weekday            44
weekday_pattern_on_weekend            43
sustained_overload                    37
gradual_drift_increase                28
power_spike                           24
multiple_high_power_simultaneous      24
impossible_appliance_combo            22
sensor_glitch                         17
gradual_drift_decrease                16
Name: count, dtype: int64

Val anomaly type counts:
 anomaly_type
normal                              765
stuck_appliance_off                  14
stuck_appliance_on                   13
appliance_unusual_hours              12
high_usage_low_occupancy             11
heating_on_warm_day  

In [6]:
X_train_normal = X_train[y_train == 0]
print(f"\nNormal training rows: {len(X_train_normal)}")

scaler = MinMaxScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

neg_val = (scaler.transform(X_val) < 0).sum()
neg_test = (scaler.transform(X_test) < 0).sum()
print(f"Clipped negative values — val: {neg_val}, test: {neg_test}")



Normal training rows: 3568
Clipped negative values — val: 205, test: 165


Basic Recon Error

In [13]:
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.linalg.norm(X_ori - X_recon, axis=1)

def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
    train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

    W_val = nmf.transform(X_val_scaled)
    X_val_recon = nmf.inverse_transform(W_val)
    val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [14]:
n_components_list = [40, 60, 80, 100, 120, 150]

results = []
models = {}
overall_start = time.perf_counter()

for n_comps in n_components_list:
    out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components=n_comps)
    results.append({
        "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
        "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
        "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
        "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
    })
    models[n_comps] = out

overall_time = time.perf_counter() - overall_start
results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
print("\nValidation results:")
print(results_df.to_string(index=False))
print(f"\nTotal sweep runtime: {overall_time:.1f}s")

# ============================================================
# CELL 8 — pick best model by PR-AUC
# ============================================================
best_k = int(results_df.iloc[0]["n_components"])
best_model = models[best_k]["model"]
best_threshold = models[best_k]["best_threshold"]

print(f"\nBest n_components = {best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")
print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

# ============================================================
# CELL 9 — final test evaluation
# ============================================================
W_test = best_model.transform(X_test_scaled)
X_test_recon = best_model.inverse_transform(W_test)
test_scores = reconstruction_error_per_sample(X_test_scaled, X_test_recon)

test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)

print("\nTest results:")
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")

# ============================================================
# CELL 10 — per-anomaly-type breakdown
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_recon = best_model.inverse_transform(best_model.transform(X_sub_scaled))
    sub_scores = reconstruction_error_per_sample(X_sub_scaled, sub_recon)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print("\n", per_type_df.to_string(index=False))

K=  40 |            OK | n_iter=  185 | fit_time=   2.3s | train_err=1.3829 | roc_auc=0.7902 | pr_auc=0.3969
K=  60 |            OK | n_iter=  844 | fit_time=  41.4s | train_err=0.9775 | roc_auc=0.7932 | pr_auc=0.3682
K=  80 |            OK | n_iter=  336 | fit_time=  27.6s | train_err=0.7583 | roc_auc=0.7972 | pr_auc=0.3868
K= 100 |            OK | n_iter=  301 | fit_time=  36.2s | train_err=0.5453 | roc_auc=0.8022 | pr_auc=0.3950
K= 120 |            OK | n_iter=  653 | fit_time=  57.4s | train_err=0.4249 | roc_auc=0.7877 | pr_auc=0.3898
K= 150 |            OK | n_iter=  510 | fit_time=  93.8s | train_err=0.3333 | roc_auc=0.7881 | pr_auc=0.3636

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
           40       True     185           2.3         1.382880 0.790230 0.396907     0.475248            0.545455         0.421053        2.574364
          100       True     

1. Ae-sad

In [17]:
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.sum((X_ori - X_recon) ** 2, axis=1)

def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
    train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

    W_val = nmf.transform(X_val_scaled)
    X_val_recon = nmf.inverse_transform(W_val)
    val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [18]:
n_components_list = [40, 60, 80, 100, 120, 150]

results = []
models = {}
overall_start = time.perf_counter()

for n_comps in n_components_list:
    out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components=n_comps)
    results.append({
        "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
        "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
        "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
        "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
    })
    models[n_comps] = out

overall_time = time.perf_counter() - overall_start
results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
print("\nValidation results:")
print(results_df.to_string(index=False))
print(f"\nTotal sweep runtime: {overall_time:.1f}s")

# ============================================================
# CELL 8 — pick best model by PR-AUC
# ============================================================
best_k = int(results_df.iloc[0]["n_components"])
best_model = models[best_k]["model"]
best_threshold = models[best_k]["best_threshold"]

print(f"\nBest n_components = {best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")
print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

# ============================================================
# CELL 9 — final test evaluation
# ============================================================
W_test = best_model.transform(X_test_scaled)
X_test_recon = best_model.inverse_transform(W_test)
test_scores = reconstruction_error_per_sample(X_test_scaled, X_test_recon)

test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)

print("\nTest results:")
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")

# ============================================================
# CELL 10 — per-anomaly-type breakdown
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_recon = best_model.inverse_transform(best_model.transform(X_sub_scaled))
    sub_scores = reconstruction_error_per_sample(X_sub_scaled, sub_recon)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print("\n", per_type_df.to_string(index=False))

K=  40 |            OK | n_iter=  185 | fit_time=   6.9s | train_err=2.1676 | roc_auc=0.7902 | pr_auc=0.3969
K=  60 |            OK | n_iter=  844 | fit_time=  22.1s | train_err=1.1603 | roc_auc=0.7932 | pr_auc=0.3682
K=  80 |            OK | n_iter=  336 | fit_time=  19.2s | train_err=0.7480 | roc_auc=0.7972 | pr_auc=0.3868
K= 100 |            OK | n_iter=  301 | fit_time=  21.8s | train_err=0.4860 | roc_auc=0.8022 | pr_auc=0.3950
K= 120 |            OK | n_iter=  653 | fit_time=  47.0s | train_err=0.3249 | roc_auc=0.7877 | pr_auc=0.3898
K= 150 |            OK | n_iter=  510 | fit_time=  82.4s | train_err=0.2223 | roc_auc=0.7881 | pr_auc=0.3636

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
           40       True     185           6.9         2.167588 0.790230 0.396907     0.475248            0.545455         0.421053        6.627596
          100       True     

2. LFR
Investigating recon gap in latent space instead of input space
latent_error = np.linalg.norm(W - W_reconstructed, axis=1)
(not exactly LFR).

In [11]:

def lfr_score(nmf, X):
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)
    W_reconstructed = nmf.transform(X_recon)
    return np.linalg.norm(W - W_reconstructed, axis=1)

def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_err = lfr_score(nmf, X_train_normal_scaled).mean()
    val_scores = lfr_score(nmf, X_val_scaled)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)


# --------------------------------------------------------

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))

    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}

    
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [12]:

n_components_list = [40, 60, 80, 100, 120, 150]

results = []
models = {}
overall_start = time.perf_counter()

for n_comps in n_components_list:
    out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components=n_comps)
    results.append({
        "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
        "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
        "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
        "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
    })
    models[n_comps] = out

overall_time = time.perf_counter() - overall_start
results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
print("\nValidation results:")
print(results_df.to_string(index=False))
print(f"\nTotal sweep runtime: {overall_time:.1f}s")

# ============================================================
# pick best model by PR-AUC
# ============================================================
best_k = int(results_df.iloc[0]["n_components"])
best_model = models[best_k]["model"]
best_threshold = models[best_k]["best_threshold"]

print(f"\nBest n_components = {best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")
print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

# ============================================================
# final test evaluation — MUST use lfr_score, not raw recon error
# ============================================================
test_scores = lfr_score(best_model, X_test_scaled)
test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)

print("\nTest results:")
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")

# ============================================================
# per-anomaly-type breakdown — same lfr_score again
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_scores = lfr_score(best_model, X_sub_scaled)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print("\n", per_type_df.to_string(index=False))

K=  40 |            OK | n_iter=  185 | fit_time=   2.4s | train_err=0.0653 | roc_auc=0.5247 | pr_auc=0.1496
K=  60 |            OK | n_iter=  844 | fit_time=  11.7s | train_err=0.0026 | roc_auc=0.4054 | pr_auc=0.1234
K=  80 |            OK | n_iter=  336 | fit_time=   7.9s | train_err=0.0414 | roc_auc=0.3314 | pr_auc=0.1071
K= 100 |            OK | n_iter=  301 | fit_time=  10.8s | train_err=0.0627 | roc_auc=0.4988 | pr_auc=0.1458
K= 120 |            OK | n_iter=  653 | fit_time=  30.4s | train_err=0.0311 | roc_auc=0.4261 | pr_auc=0.1192
K= 150 |            OK | n_iter=  510 | fit_time=  36.5s | train_err=0.0442 | roc_auc=0.5259 | pr_auc=0.1865

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
          150       True     510          36.5         0.044243 0.525926 0.186508     0.305882            0.276596         0.342105    6.202987e-02
           40       True     

3. RGAnomaly
Multiple recon signals combined
error = alpha * input_error + (1 - alpha) * latent_error


In [21]:
alpha = 0.5

In [9]:
def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    # train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
    # train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

    # W_val = nmf.transform(X_val_scaled)
    # X_val_recon = nmf.inverse_transform(W_val)
    # val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

    # LFR: Latent-Space Reconstruction Gap

    W_train = nmf.transform(X_train_normal_scaled)
    X_train_recon = nmf.inverse_transform(W_train)

    # Input-space reconstruction error
    input_error_train = np.linalg.norm(
        X_train_normal_scaled - X_train_recon,
        axis=1
    )



    W_train_reconstructed = nmf.transform(X_train_recon)

    # Latent-space reconstruction error
    latent_error_train = np.linalg.norm(
        W_train - W_train_reconstructed,
        axis=1
    )

    # Combined RGAnomaly error
    train_err = (
        alpha * input_error_train
        + (1 - alpha) * latent_error_train
    ).mean()


    W_val = nmf.transform(X_val_scaled)

    X_val_recon = nmf.inverse_transform(W_val)

    # Input-space reconstruction error
    input_error = np.linalg.norm(
        X_val_scaled - X_val_recon,
        axis=1
    )

    # Reconstruct latent representation
    W_val_reconstructed = nmf.transform(X_val_recon)

    # Latent-space reconstruction error
    latent_error = np.linalg.norm(
        W_val - W_val_reconstructed,
        axis=1
    )

    # RGAnomaly score
    val_scores = (
        alpha * input_error
        + (1 - alpha) * latent_error
    )

# --------------------------------------------------------

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))

    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}

    
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
    }

In [10]:
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.sum((X_ori - X_recon) ** 2, axis=1)


def rganomaly_score(nmf, X, alpha):
    """
    alpha * input-space recon error + (1 - alpha) * latent-space recon error.
    """
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)

    # input-space error
    input_error = np.linalg.norm(X - X_recon, axis=1)

    # latent-space error: re-encode the reconstruction, compare to original W
    W_reconstructed = nmf.transform(X_recon)
    latent_error = np.linalg.norm(W - W_reconstructed, axis=1)

    return alpha * input_error + (1 - alpha) * latent_error


def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
                       alpha=0.5, max_iter=1500, tol=1e-3, random_state=42):
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_components,
            init="nndsvda",
            solver="cd",
            max_iter=max_iter,
            tol=tol,
            random_state=random_state,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_scores = rganomaly_score(nmf, X_train_normal_scaled, alpha)
    train_err = train_scores.mean()

    val_scores = rganomaly_score(nmf, X_val_scaled, alpha)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (val_scores >= t).astype(int)
        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

    print(f"K={n_components:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | "
          f"train_err={train_err:.4f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f}")

    return {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
        "roc_auc": roc_auc, "pr_auc": pr_auc,
        "best_threshold": best["threshold"], "best_val_f1": best["f1"],
        "best_val_precision": best["precision"], "best_val_recall": best["recall"],
        "val_scores": val_scores,
        "alpha": alpha,
    }


alphas = [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
n_components_list = [40, 60, 80, 100, 120, 150]

all_alpha_summaries = []

for alpha in alphas:
    print(f"\n{'='*60}\nAlpha = {alpha}\n{'='*60}")

    results = []
    models = {}
    overall_start = time.perf_counter()

    for n_comps in n_components_list:
        out = fit_and_score_nmf(
            X_train_normal_scaled, X_val_scaled, y_val,
            n_components=n_comps, alpha=alpha
        )
        results.append({
            "n_components": n_comps, "converged": out["converged"], "n_iter": out["n_iter"],
            "fit_time_sec": round(out["fit_time"], 1), "train_recon_err": out["train_recon_err"],
            "roc_auc": out["roc_auc"], "pr_auc": out["pr_auc"],
            "best_val_f1": out["best_val_f1"], "best_val_precision": out["best_val_precision"],
            "best_val_recall": out["best_val_recall"], "best_threshold": out["best_threshold"],
        })
        models[n_comps] = out

    overall_time = time.perf_counter() - overall_start
    results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
    print("\nValidation results:")
    print(results_df.to_string(index=False))
    print(f"\nTotal sweep runtime: {overall_time:.1f}s")

    # ------------------------------------------------------------
    # pick best model by PR-AUC
    # ------------------------------------------------------------
    best_k = int(results_df.iloc[0]["n_components"])
    best_model = models[best_k]["model"]
    best_threshold = models[best_k]["best_threshold"]

    print(f"\nBest n_components = {best_k}")
    print(f"Best validation threshold = {best_threshold:.6f}")
    print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

    # ------------------------------------------------------------
    # final test evaluation — MUST use the same scoring fn as training/val
    # ------------------------------------------------------------
    test_scores = rganomaly_score(best_model, X_test_scaled, alpha)
    test_pred = (test_scores >= best_threshold).astype(int)

    test_roc_auc = roc_auc_score(y_test, test_scores)
    test_pr_auc = average_precision_score(y_test, test_scores)
    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)

    print("\nTest results:")
    print(f"ROC AUC:    {test_roc_auc:.4f}")
    print(f"PR AUC:     {test_pr_auc:.4f}")
    print(f"Precision:  {test_precision:.4f}")
    print(f"Recall:     {test_recall:.4f}")
    print(f"F1:         {test_f1:.4f}")

    # ------------------------------------------------------------
    # per-anomaly-type breakdown — same scoring fn again
    # ------------------------------------------------------------
    print("\nPer-anomaly-type AUC on test set:")
    per_type_results = []
    for atype in sorted(test_df['anomaly_type'].unique()):
        if atype == 'normal':
            continue
        mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
        sub = test_df[mask]
        X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
        sub_scores = rganomaly_score(best_model, X_sub_scaled, alpha)
        try:
            auc = roc_auc_score(sub['is_anomaly'], sub_scores)
        except ValueError:
            auc = float('nan')
        n_pos = (sub['is_anomaly'] == 1).sum()
        print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
        per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

    per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
    print("\n", per_type_df.to_string(index=False))

    all_alpha_summaries.append({
        "alpha": alpha, "best_k": best_k,
        "test_roc_auc": test_roc_auc, "test_pr_auc": test_pr_auc,
        "test_f1": test_f1, "test_precision": test_precision, "test_recall": test_recall,
    })

print(f"\n{'='*60}\nSummary across alphas\n{'='*60}")
summary_df = pd.DataFrame(all_alpha_summaries).sort_values("test_pr_auc", ascending=False)
print(summary_df.to_string(index=False))


Alpha = 0.2
K=  40 |            OK | n_iter=  185 | fit_time=   2.3s | train_err=0.3288 | roc_auc=0.7787 | pr_auc=0.4052
K=  60 |            OK | n_iter=  844 | fit_time=  12.0s | train_err=0.1976 | roc_auc=0.7932 | pr_auc=0.3687
K=  80 |            OK | n_iter=  336 | fit_time=   8.0s | train_err=0.1848 | roc_auc=0.7863 | pr_auc=0.3878
K= 100 |            OK | n_iter=  301 | fit_time=  10.3s | train_err=0.1592 | roc_auc=0.7956 | pr_auc=0.3747
K= 120 |            OK | n_iter=  653 | fit_time=  28.1s | train_err=0.1099 | roc_auc=0.7824 | pr_auc=0.3844
K= 150 |            OK | n_iter=  510 | fit_time=  37.3s | train_err=0.1020 | roc_auc=0.7756 | pr_auc=0.3469

Validation results:
 n_components  converged  n_iter  fit_time_sec  train_recon_err  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
           40       True     185           2.3         0.328810 0.778730 0.405169     0.484536            0.587500         0.412281        0.572380
           80   

Best Model - RGAnomaly - Run 1


In [1]:
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

# ============================================================
# BEST CONFIG — locked in from the alpha/K sweep
# ============================================================
BEST_ALPHA = 0.4
BEST_K = 40

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(r'pipeline_cache\ampds_behavior_context_labeled_features.csv')
df = df.drop(columns=['window_id'])

# ============================================================
# STRATIFIED SPLIT — same split used throughout, for consistency
# ============================================================
df_reset = df.reset_index(drop=True)
anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
normal_df = df_reset[df_reset['is_anomaly'] == 0]

anom_train, anom_temp = train_test_split(
    anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
)
anom_val, anom_test = train_test_split(
    anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
)
norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

drop_cols = ['is_anomaly', 'anomaly_type']
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_anomaly']
X_val = val_df.drop(columns=drop_cols)
y_val = val_df['is_anomaly']
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['is_anomaly']

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# ============================================================
# TRAIN ON NORMAL DATA ONLY, SCALE
# ============================================================
X_train_normal = X_train[y_train == 0]
print(f"Normal training rows: {len(X_train_normal)}")

scaler = MinMaxScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

# ============================================================
# SCORING FUNCTION — combined input-space + latent-space reconstruction error
# ============================================================
def rganomaly_score(nmf, X, alpha):
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)
    input_error = np.linalg.norm(X - X_recon, axis=1)

    W_reconstructed = nmf.transform(X_recon)
    latent_error = np.linalg.norm(W - W_reconstructed, axis=1)

    return alpha * input_error + (1 - alpha) * latent_error

# ============================================================
# FIT — raw, tight convergence, no efficiency shortcuts (single run, cost is trivial)
# ============================================================
print(f"\nFitting NMF: K={BEST_K}, alpha={BEST_ALPHA}")

start = time.perf_counter()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    nmf = NMF(
        n_components=BEST_K,
        init="nndsvda",
        solver="cd",
        max_iter=3000,               
        random_state=42,
    )
    nmf.fit(X_train_normal_scaled)
    converged = len(w) == 0
fit_time = time.perf_counter() - start

print(f"Converged: {converged} | n_iter={nmf.n_iter_}/{nmf.max_iter} | fit_time={fit_time:.2f}s")

# ============================================================
# VALIDATION — threshold selection only
# ============================================================
val_scores = rganomaly_score(nmf, X_val_scaled, BEST_ALPHA)
val_roc_auc = roc_auc_score(y_val, val_scores)
val_pr_auc = average_precision_score(y_val, val_scores)

thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
for t in thresholds:
    preds = (val_scores >= t).astype(int)
    p = precision_score(y_val, preds, zero_division=0)
    r = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best["f1"]:
        best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

print(f"\nValidation: ROC-AUC={val_roc_auc:.4f} | PR-AUC={val_pr_auc:.4f} | "
      f"Best F1={best['f1']:.4f} @ threshold={best['threshold']:.6f}")

# ============================================================
# FINAL TEST EVALUATION
# ============================================================
test_scores = rganomaly_score(nmf, X_test_scaled, BEST_ALPHA)
test_pred = (test_scores >= best["threshold"]).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print("\n" + "="*60)
print(f"FINAL RESULT — NMF (K={BEST_K}, alpha={BEST_ALPHA})")
print("="*60)
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")
print(f"Confusion:  TP={tp}, FP={fp}, FN={fn}, TN={tn}")

# ============================================================
# PER-ANOMALY-TYPE BREAKDOWN
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_scores = rganomaly_score(nmf, X_sub_scaled, BEST_ALPHA)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print(per_type_df.to_string(index=False))

Train: (4098, 830) | Val: (879, 830) | Test: (879, 830)
Normal training rows: 3568

Fitting NMF: K=40, alpha=0.4
Converged: True | n_iter=821/3000 | fit_time=9.69s

Validation: ROC-AUC=0.7817 | PR-AUC=0.3977 | Best F1=0.4818 @ threshold=0.851208

FINAL RESULT — NMF (K=40, alpha=0.4)
ROC AUC:    0.7702
PR AUC:     0.4024
Precision:  0.4898
Recall:     0.4211
F1:         0.4528
Confusion:  TP=48, FP=50, FN=66, TN=715

Per-anomaly-type AUC on test set:
                    anomaly_type      auc  n_anomaly
      weekend_pattern_on_weekday 0.984604          9
      weekday_pattern_on_weekend 0.978794          9
             stuck_appliance_off 0.976370         13
                   sensor_glitch 0.964052          4
      impossible_appliance_combo 0.957386          5
              sustained_overload 0.854739          8
multiple_high_power_simultaneous 0.832157          5
                     power_spike 0.758170          5
             heating_on_warm_day 0.755701          9
          gradua

In [2]:
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

# ============================================================
# BEST CONFIG — locked in from the alpha/K sweep
# ============================================================
BEST_ALPHA = 0.5
BEST_K = 40

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(r'pipeline_cache\ampds_behavior_context_labeled_features.csv')
df = df.drop(columns=['window_id'])

# ============================================================
# STRATIFIED SPLIT — same split used throughout, for consistency
# ============================================================
df_reset = df.reset_index(drop=True)
anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
normal_df = df_reset[df_reset['is_anomaly'] == 0]

anom_train, anom_temp = train_test_split(
    anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
)
anom_val, anom_test = train_test_split(
    anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
)
norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

drop_cols = ['is_anomaly', 'anomaly_type']
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_anomaly']
X_val = val_df.drop(columns=drop_cols)
y_val = val_df['is_anomaly']
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['is_anomaly']

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# ============================================================
# TRAIN ON NORMAL DATA ONLY, SCALE
# ============================================================
X_train_normal = X_train[y_train == 0]
print(f"Normal training rows: {len(X_train_normal)}")

scaler = MinMaxScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

# ============================================================
# SCORING FUNCTION — combined input-space + latent-space reconstruction error
# ============================================================
def rganomaly_score(nmf, X, alpha):
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)
    input_error = np.linalg.norm(X - X_recon, axis=1)

    W_reconstructed = nmf.transform(X_recon)
    latent_error = np.linalg.norm(W - W_reconstructed, axis=1)

    return alpha * input_error + (1 - alpha) * latent_error

# ============================================================
# FIT — raw, tight convergence, no efficiency shortcuts (single run, cost is trivial)
# ============================================================
print(f"\nFitting NMF: K={BEST_K}, alpha={BEST_ALPHA}")

start = time.perf_counter()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    nmf = NMF(
        n_components=BEST_K,
        init="nndsvda",
        solver="cd",
        max_iter=3000,               
        random_state=42,
    )
    nmf.fit(X_train_normal_scaled)
    converged = len(w) == 0
fit_time = time.perf_counter() - start

print(f"Converged: {converged} | n_iter={nmf.n_iter_}/{nmf.max_iter} | fit_time={fit_time:.2f}s")

# ============================================================
# VALIDATION — threshold selection only
# ============================================================
val_scores = rganomaly_score(nmf, X_val_scaled, BEST_ALPHA)
val_roc_auc = roc_auc_score(y_val, val_scores)
val_pr_auc = average_precision_score(y_val, val_scores)

thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
for t in thresholds:
    preds = (val_scores >= t).astype(int)
    p = precision_score(y_val, preds, zero_division=0)
    r = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best["f1"]:
        best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

print(f"\nValidation: ROC-AUC={val_roc_auc:.4f} | PR-AUC={val_pr_auc:.4f} | "
      f"Best F1={best['f1']:.4f} @ threshold={best['threshold']:.6f}")

# ============================================================
# FINAL TEST EVALUATION
# ============================================================
test_scores = rganomaly_score(nmf, X_test_scaled, BEST_ALPHA)
test_pred = (test_scores >= best["threshold"]).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print("\n" + "="*60)
print(f"FINAL RESULT — NMF (K={BEST_K}, alpha={BEST_ALPHA})")
print("="*60)
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")
print(f"Confusion:  TP={tp}, FP={fp}, FN={fn}, TN={tn}")

# ============================================================
# PER-ANOMALY-TYPE BREAKDOWN
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_scores = rganomaly_score(nmf, X_sub_scaled, BEST_ALPHA)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print(per_type_df.to_string(index=False))

Train: (4098, 830) | Val: (879, 830) | Test: (879, 830)
Normal training rows: 3568

Fitting NMF: K=40, alpha=0.5
Converged: True | n_iter=821/3000 | fit_time=7.24s

Validation: ROC-AUC=0.7815 | PR-AUC=0.3977 | Best F1=0.4818 @ threshold=1.063913

FINAL RESULT — NMF (K=40, alpha=0.5)
ROC AUC:    0.7699
PR AUC:     0.4023
Precision:  0.4898
Recall:     0.4211
F1:         0.4528
Confusion:  TP=48, FP=50, FN=66, TN=715

Per-anomaly-type AUC on test set:
                    anomaly_type      auc  n_anomaly
      weekend_pattern_on_weekday 0.984604          9
      weekday_pattern_on_weekend 0.978794          9
             stuck_appliance_off 0.976471         13
                   sensor_glitch 0.964052          4
      impossible_appliance_combo 0.957386          5
              sustained_overload 0.854739          8
multiple_high_power_simultaneous 0.832157          5
                     power_spike 0.757647          5
             heating_on_warm_day 0.754829          9
          gradua

Time - added contextually - run

In [1]:
import time
import warnings

import numpy as np
import pandas as pd

from sklearn.decomposition import NMF
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler


# ============================================================
# CONFIG
# ============================================================

BEST_ALPHA = 0.4
BEST_K = 40

RANDOM_STATE = 42

DATA_PATH = (
    r"pipeline_cache\ampds_behavior_context_labeled_features.csv"
)


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

# Keep window_id because it contains the timestamp.
# It will be used to create time features, but the raw
# timestamp itself will NOT be given to NMF.

df["window_id"] = pd.to_datetime(
    df["window_id"],
    utc=True,
    errors="coerce",
)

if df["window_id"].isna().any():
    raise ValueError(
        "Some window_id values could not be parsed as timestamps."
    )


# ============================================================
# ADD BASIC TIME CONTEXT
# ============================================================
#
# Your data is in 15-minute windows.
#
# We therefore represent:
#
#   1. Time of day
#   2. 15-minute position within the day
#   3. Day of week
#   4. Weekend / weekday
#
# We use sin/cos because time is cyclic:
#
#   23:45 -> 00:00
#
# should be considered close together.
# ============================================================

timestamp = df["window_id"]

hour = timestamp.dt.hour
minute = timestamp.dt.minute
dow = timestamp.dt.dayofweek

# ------------------------------------------------------------
# 1. Hour of day
# ------------------------------------------------------------

hour_decimal = (
    hour
    + minute / 60.0
)

df["hour_sin"] = np.sin(
    2 * np.pi * hour_decimal / 24.0
)

df["hour_cos"] = np.cos(
    2 * np.pi * hour_decimal / 24.0
)


# ------------------------------------------------------------
# 2. 15-minute slot of day
# ------------------------------------------------------------
#
# 00:00 -> 0
# 00:15 -> 1
# ...
# 23:45 -> 95
# ------------------------------------------------------------

slot_of_day = (
    hour * 4
    + minute // 15
)

df["slot_sin"] = np.sin(
    2 * np.pi * slot_of_day / 96.0
)

df["slot_cos"] = np.cos(
    2 * np.pi * slot_of_day / 96.0
)


# ------------------------------------------------------------
# 3. Day of week
# ------------------------------------------------------------
#
# Monday    = 0
# Tuesday   = 1
# ...
# Sunday    = 6
# ------------------------------------------------------------

df["dow_sin"] = np.sin(
    2 * np.pi * dow / 7.0
)

df["dow_cos"] = np.cos(
    2 * np.pi * dow / 7.0
)


# ------------------------------------------------------------
# 4. Weekend indicator
# ------------------------------------------------------------

df["is_weekend"] = (
    dow >= 5
).astype(int)


# ============================================================
# SPLIT DATA
# ============================================================
#
# We keep the exact same splitting strategy as your original
# model so the comparison remains fair.
#
# Anomalies are stratified by anomaly_type.
# ============================================================

df = df.reset_index(drop=True)

anomaly_df = df[
    df["is_anomaly"] == 1
].copy()

normal_df = df[
    df["is_anomaly"] == 0
].copy()


# ------------------------------------------------------------
# Anomaly split
# ------------------------------------------------------------

anom_train, anom_temp = train_test_split(
    anomaly_df,
    test_size=0.30,
    stratify=anomaly_df["anomaly_type"],
    random_state=RANDOM_STATE,
)

anom_val, anom_test = train_test_split(
    anom_temp,
    test_size=0.50,
    stratify=anom_temp["anomaly_type"],
    random_state=RANDOM_STATE,
)


# ------------------------------------------------------------
# Normal split
# ------------------------------------------------------------

norm_train, norm_temp = train_test_split(
    normal_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

norm_val, norm_test = train_test_split(
    norm_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
)


# ------------------------------------------------------------
# Combine and shuffle
# ------------------------------------------------------------

train_df = (
    pd.concat(
        [anom_train, norm_train]
    )
    .sample(
        frac=1,
        random_state=RANDOM_STATE,
    )
    .reset_index(drop=True)
)

val_df = (
    pd.concat(
        [anom_val, norm_val]
    )
    .sample(
        frac=1,
        random_state=RANDOM_STATE,
    )
    .reset_index(drop=True)
)

test_df = (
    pd.concat(
        [anom_test, norm_test]
    )
    .sample(
        frac=1,
        random_state=RANDOM_STATE,
    )
    .reset_index(drop=True)
)


print(
    f"Train: {train_df.shape} | "
    f"Val: {val_df.shape} | "
    f"Test: {test_df.shape}"
)


# ============================================================
# DEFINE MODEL FEATURES
# ============================================================
#
# We remove:
#
#   window_id
#   is_anomaly
#   anomaly_type
#
# Everything else becomes an input feature.
#
# This means the model gets:
#
#   original behavior features
#   +
#   7 basic time/context features
#
# No manually-created contextual z-scores.
# ============================================================

DROP_COLUMNS = [
    "window_id",
    "is_anomaly",
    "anomaly_type",
]


X_train = train_df.drop(
    columns=DROP_COLUMNS
)

y_train = train_df["is_anomaly"].astype(int)

X_val = val_df.drop(
    columns=DROP_COLUMNS
)

y_val = val_df["is_anomaly"].astype(int)

X_test = test_df.drop(
    columns=DROP_COLUMNS
)

y_test = test_df["is_anomaly"].astype(int)


print(
    f"Total features: {X_train.shape[1]}"
)

print(
    "Added time features:"
)

print(
    "  - hour_sin"
)

print(
    "  - hour_cos"
)

print(
    "  - slot_sin"
)

print(
    "  - slot_cos"
)

print(
    "  - dow_sin"
)

print(
    "  - dow_cos"
)

print(
    "  - is_weekend"
)


# ============================================================
# TRAIN ON NORMAL DATA ONLY
# ============================================================

X_train_normal = X_train[
    y_train == 0
].copy()

print(
    f"\nNormal training rows: "
    f"{len(X_train_normal)}"
)


# ============================================================
# SCALE FEATURES
# ============================================================
#
# The scaler is fitted ONLY on normal training data.
#
# This preserves the anomaly-detection setup:
#
#   normal train -> fit scaler
#   normal train -> fit NMF
#   validation   -> transform only
#   test         -> transform only
# ============================================================

scaler = MinMaxScaler()

X_train_normal_scaled = scaler.fit_transform(
    X_train_normal
)

X_val_scaled = scaler.transform(
    X_val
)

X_test_scaled = scaler.transform(
    X_test
)


# ------------------------------------------------------------
# Safety: NMF requires non-negative values
# ------------------------------------------------------------

X_train_normal_scaled = np.clip(
    X_train_normal_scaled,
    0,
    None,
)

X_val_scaled = np.clip(
    X_val_scaled,
    0,
    None,
)

X_test_scaled = np.clip(
    X_test_scaled,
    0,
    None,
)


# ============================================================
# NMF ANOMALY SCORE
# ============================================================

def nmf_anomaly_score(
    nmf,
    X,
    alpha,
):
    """
    Combined input-space + latent-space reconstruction error.
    """

    # --------------------------------------------------------
    # Input -> latent representation
    # --------------------------------------------------------

    W = nmf.transform(X)

    # --------------------------------------------------------
    # Latent -> reconstructed input
    # --------------------------------------------------------

    X_reconstructed = nmf.inverse_transform(W)

    # --------------------------------------------------------
    # Input-space reconstruction error
    # --------------------------------------------------------

    input_error = np.linalg.norm(
        X - X_reconstructed,
        axis=1,
    )

    # --------------------------------------------------------
    # Reconstruct latent representation
    # --------------------------------------------------------

    W_reconstructed = nmf.transform(
        X_reconstructed
    )

    # --------------------------------------------------------
    # Latent-space reconstruction error
    # --------------------------------------------------------

    latent_error = np.linalg.norm(
        W - W_reconstructed,
        axis=1,
    )

    # --------------------------------------------------------
    # Combined anomaly score
    # --------------------------------------------------------

    return (
        alpha * input_error
        + (1.0 - alpha) * latent_error
    )


# ============================================================
# FIT NMF
# ============================================================

print(
    f"\nFitting Time-Aware NMF: "
    f"K={BEST_K}, alpha={BEST_ALPHA}"
)

start_time = time.perf_counter()

with warnings.catch_warnings(
    record=True
) as warning_list:

    warnings.simplefilter(
        "always"
    )

    nmf = NMF(
        n_components=BEST_K,
        init="nndsvda",
        solver="cd",
        max_iter=3000,
        random_state=RANDOM_STATE,
    )

    nmf.fit(
        X_train_normal_scaled
    )

    converged = len(warning_list) == 0

fit_time = (
    time.perf_counter()
    - start_time
)

print(
    f"Converged: {converged} | "
    f"n_iter={nmf.n_iter_}/{nmf.max_iter} | "
    f"fit_time={fit_time:.2f}s"
)


# ============================================================
# VALIDATION
# ============================================================
#
# Validation is used to:
#
#   1. Measure ranking performance
#   2. Select the anomaly threshold
#
# The test set is NOT used here.
# ============================================================

val_scores = nmf_anomaly_score(
    nmf,
    X_val_scaled,
    BEST_ALPHA,
)

val_roc_auc = roc_auc_score(
    y_val,
    val_scores,
)

val_pr_auc = average_precision_score(
    y_val,
    val_scores,
)


# ============================================================
# SELECT BEST F1 THRESHOLD
# ============================================================

thresholds = np.unique(
    np.quantile(
        val_scores,
        np.linspace(
            0.01,
            0.99,
            99,
        ),
    )
)

best = {
    "threshold": None,
    "f1": -1.0,
    "precision": None,
    "recall": None,
}


for threshold in thresholds:

    predictions = (
        val_scores >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        predictions,
        zero_division=0,
    )

    recall = recall_score(
        y_val,
        predictions,
        zero_division=0,
    )

    f1 = f1_score(
        y_val,
        predictions,
        zero_division=0,
    )

    if f1 > best["f1"]:

        best = {
            "threshold": float(threshold),
            "f1": float(f1),
            "precision": float(precision),
            "recall": float(recall),
        }


print(
    f"\nValidation:"
    f" ROC-AUC={val_roc_auc:.4f}"
    f" | PR-AUC={val_pr_auc:.4f}"
    f" | Best F1={best['f1']:.4f}"
    f" | Threshold={best['threshold']:.6f}"
)


# ============================================================
# FINAL TEST EVALUATION
# ============================================================

test_scores = nmf_anomaly_score(
    nmf,
    X_test_scaled,
    BEST_ALPHA,
)

test_predictions = (
    test_scores >= best["threshold"]
).astype(int)


# ------------------------------------------------------------
# Ranking metrics
# ------------------------------------------------------------

test_roc_auc = roc_auc_score(
    y_test,
    test_scores,
)

test_pr_auc = average_precision_score(
    y_test,
    test_scores,
)


# ------------------------------------------------------------
# Threshold metrics
# ------------------------------------------------------------

test_precision = precision_score(
    y_test,
    test_predictions,
    zero_division=0,
)

test_recall = recall_score(
    y_test,
    test_predictions,
    zero_division=0,
)

test_f1 = f1_score(
    y_test,
    test_predictions,
    zero_division=0,
)


# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

tn, fp, fn, tp = confusion_matrix(
    y_test,
    test_predictions,
).ravel()


# ============================================================
# FINAL RESULTS
# ============================================================

print("\n" + "=" * 65)

print(
    f"FINAL RESULT — TIME-AWARE NMF "
    f"(K={BEST_K}, alpha={BEST_ALPHA})"
)

print("=" * 65)

print(
    f"ROC AUC:    {test_roc_auc:.4f}"
)

print(
    f"PR AUC:     {test_pr_auc:.4f}"
)

print(
    f"Precision:  {test_precision:.4f}"
)

print(
    f"Recall:     {test_recall:.4f}"
)

print(
    f"F1:         {test_f1:.4f}"
)

print(
    f"Confusion:  "
    f"TP={tp}, FP={fp}, "
    f"FN={fn}, TN={tn}"
)


# ============================================================
# PER-ANOMALY-TYPE AUC
# ============================================================

print(
    "\nPer-anomaly-type AUC on test set:"
)

per_type_results = []


for anomaly_type in sorted(
    test_df["anomaly_type"].unique()
):

    if anomaly_type == "normal":
        continue

    # --------------------------------------------------------
    # Current anomaly type + all normal rows
    # --------------------------------------------------------

    mask = (
        (test_df["anomaly_type"] == anomaly_type)
        | (test_df["is_anomaly"] == 0)
    )

    subset = test_df.loc[
        mask
    ].copy()

    X_subset = subset.drop(
        columns=DROP_COLUMNS
    )

    X_subset_scaled = scaler.transform(
        X_subset
    )

    X_subset_scaled = np.clip(
        X_subset_scaled,
        0,
        None,
    )

    subset_scores = nmf_anomaly_score(
        nmf,
        X_subset_scaled,
        BEST_ALPHA,
    )

    try:

        auc = roc_auc_score(
            subset["is_anomaly"],
            subset_scores,
        )

    except ValueError:

        auc = np.nan

    n_anomalies = (
        subset["is_anomaly"] == 1
    ).sum()

    per_type_results.append(
        {
            "anomaly_type": anomaly_type,
            "auc": auc,
            "n_anomaly": n_anomalies,
        }
    )


per_type_df = (
    pd.DataFrame(
        per_type_results
    )
    .sort_values(
        "auc",
        ascending=False,
    )
)


print(
    per_type_df.to_string(
        index=False
    )
)


# ============================================================
# TIME FEATURE SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("TIME CONTEXT SUMMARY")
print("=" * 65)

print(
    "Time features added: 7"
)

print(
    "  hour_sin"
)

print(
    "  hour_cos"
)

print(
    "  slot_sin"
)

print(
    "  slot_cos"
)

print(
    "  dow_sin"
)

print(
    "  dow_cos"
)

print(
    "  is_weekend"
)

print(
    f"\nOriginal feature count: "
    f"{X_train.shape[1] - 7}"
)

print(
    f"Final feature count: "
    f"{X_train.shape[1]}"
)

print(
    "\nTime is provided directly to NMF."
)

print(
    "No manually-created contextual baseline "
    "or duplicated feature set is used."
)

C:\Users\revan\AppData\Local\Temp\ipykernel_24088\2645776201.py:115: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["slot_sin"] = np.sin(
C:\Users\revan\AppData\Local\Temp\ipykernel_24088\2645776201.py:119: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["slot_cos"] = np.cos(


Train: (4098, 835) | Val: (879, 835) | Test: (879, 835)
Total features: 832
Added time features:
  - hour_sin
  - hour_cos
  - slot_sin
  - slot_cos
  - dow_sin
  - dow_cos
  - is_weekend

Normal training rows: 3568

Fitting Time-Aware NMF: K=40, alpha=0.4
Converged: True | n_iter=1287/3000 | fit_time=14.98s

Validation: ROC-AUC=0.7842 | PR-AUC=0.3777 | Best F1=0.4636 | Threshold=0.859015

FINAL RESULT — TIME-AWARE NMF (K=40, alpha=0.4)
ROC AUC:    0.7636
PR AUC:     0.3827
Precision:  0.4842
Recall:     0.4035
F1:         0.4402
Confusion:  TP=46, FP=49, FN=68, TN=716

Per-anomaly-type AUC on test set:
                    anomaly_type      auc  n_anomaly
      weekend_pattern_on_weekday 0.984459          9
      weekday_pattern_on_weekend 0.978940          9
             stuck_appliance_off 0.973856         13
                   sensor_glitch 0.962092          4
      impossible_appliance_combo 0.953725          5
              sustained_overload 0.853758          8
             heati

Clustering?

In [3]:
import numpy as np
import pandas as pd
from sklearn.neural_network import BernoulliRBM
from sklearn.pipeline import Pipeline
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler

# ============================================================
# STEP 1 — pull only the NMF-flagged anomalies (test set, predicted positive)
# ============================================================
# test_pred and test_scores come from your final NMF run
flagged_mask = test_predictions == 1
X_flagged = X_test_scaled[flagged_mask]                 # already 0-1 scaled, RBM needs this
flagged_true_types = test_df.loc[flagged_mask, 'anomaly_type'].values
print(f"NMF flagged {flagged_mask.sum()} windows as anomalous")

# ============================================================
# STEP 2 — DBN: stack of RBMs, pretrained layer by layer (sklearn Pipeline)
# mirrors the dbn-based-nids repo's layer-wise RBM stack, minus the classifier head
# ============================================================
rbm1 = BernoulliRBM(n_components=128, learning_rate=0.05, n_iter=30, random_state=42)
rbm2 = BernoulliRBM(n_components=32, learning_rate=0.05, n_iter=30, random_state=42)

dbn = Pipeline(steps=[('rbm1', rbm1), ('rbm2', rbm2)])
dbn.fit(X_flagged)

latent = dbn.transform(X_flagged)     # low-dim representation of the anomalies, shape (n_flagged, 32)
print(f"Latent shape: {latent.shape}")

# ============================================================
# STEP 3 — GMM clustering on the latent space (the DAGMM idea — cluster, don't classify)
# ============================================================
n_clusters = 6   # start near your anomaly_type count, tune from here
gmm = GaussianMixture(n_components=n_clusters, covariance_type='full', random_state=42)
cluster_labels = gmm.fit_predict(latent)

# ============================================================
# STEP 4 — sanity check: do clusters line up with your known anomaly_type labels?
# ============================================================
comparison = pd.DataFrame({'true_type': flagged_true_types, 'cluster': cluster_labels})
print(pd.crosstab(comparison['true_type'], comparison['cluster']))

NMF flagged 95 windows as anomalous
Latent shape: (95, 32)
cluster                            0  1   2  3  4   5
true_type                                            
appliance_unusual_hours            0  1   0  0  0   0
gradual_drift_decrease             1  0   0  0  0   0
gradual_drift_increase             0  0   0  0  0   1
heating_on_warm_day                0  1   1  0  0   0
impossible_appliance_combo         0  1   3  0  0   0
multiple_high_power_simultaneous   0  1   0  0  0   0
normal                            17  4  11  1  0  16
sensor_glitch                      0  0   4  0  0   0
stuck_appliance_off                0  9   1  0  0   2
stuck_appliance_on                 0  0   0  0  0   1
sustained_overload                 0  0   1  0  0   0
weekday_pattern_on_weekend         8  0   0  0  1   0
weekend_pattern_on_weekday         7  0   0  0  2   0


Clustering run 2 

In [2]:
# ============================================================
# APPEND THIS AFTER THE NMF SCRIPT ABOVE — reuses nmf, test_pred, X_test_scaled, test_df
# ============================================================
from sklearn.neural_network import BernoulliRBM
from sklearn.pipeline import Pipeline
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

# ============================================================
# STEP 0 — pull out only what NMF flagged as anomalous
# ============================================================
flagged_mask = test_pred == 1
X_flagged = X_test_scaled[flagged_mask]
flagged_true_types = test_df.loc[flagged_mask, 'anomaly_type'].values

n_true_types = len(set(flagged_true_types) - {'normal'})
print(f"\nNMF flagged {flagged_mask.sum()} windows | "
      f"{len(set(flagged_true_types))} distinct true labels present (incl. false positives labeled 'normal')")

# number of clusters to aim for — start near the count of true anomaly types actually present
N_CLUSTERS = max(n_true_types, 2)
print(f"Using N_CLUSTERS = {N_CLUSTERS}")

# ============================================================
# 1. CLUSTERING DBN — stacked RBMs (compression) -> GMM (clustering on latent space)
# ============================================================
print("\n" + "="*60)
print("1. DBN + GMM clustering")
print("="*60)

rbm1 = BernoulliRBM(n_components=128, learning_rate=0.05, n_iter=50, random_state=42)
rbm2 = BernoulliRBM(n_components=32, learning_rate=0.05, n_iter=50, random_state=42)

dbn = Pipeline(steps=[('rbm1', rbm1), ('rbm2', rbm2)])
dbn.fit(X_flagged)

latent = dbn.transform(X_flagged)
print(f"Latent shape after DBN: {latent.shape}")

gmm = GaussianMixture(n_components=N_CLUSTERS, covariance_type='full', random_state=42)
dbn_cluster_labels = gmm.fit_predict(latent)

dbn_sil = silhouette_score(latent, dbn_cluster_labels) if len(set(dbn_cluster_labels)) > 1 else float('nan')
dbn_ari = adjusted_rand_score(flagged_true_types, dbn_cluster_labels)

print(f"Silhouette score (latent space): {dbn_sil:.4f}")
print(f"Adjusted Rand Index vs true anomaly_type: {dbn_ari:.4f}")
print("\nDBN+GMM cluster vs true anomaly_type:")
print(pd.crosstab(pd.Series(flagged_true_types, name='true_type'),
                   pd.Series(dbn_cluster_labels, name='dbn_cluster')))

# ============================================================
# 2. BASIC CLUSTERING MODEL — KMeans directly on raw scaled features (no compression)
# ============================================================
print("\n" + "="*60)
print("2. Basic KMeans clustering (raw feature space)")
print("="*60)

kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_flagged)

kmeans_sil = silhouette_score(X_flagged, kmeans_labels) if len(set(kmeans_labels)) > 1 else float('nan')
kmeans_ari = adjusted_rand_score(flagged_true_types, kmeans_labels)

print(f"Silhouette score (raw feature space): {kmeans_sil:.4f}")
print(f"Adjusted Rand Index vs true anomaly_type: {kmeans_ari:.4f}")
print("\nKMeans cluster vs true anomaly_type:")
print(pd.crosstab(pd.Series(flagged_true_types, name='true_type'),
                   pd.Series(kmeans_labels, name='kmeans_cluster')))

# ============================================================
# COMPARISON — which clustering approach actually separated the anomaly types better
# ============================================================
print("\n" + "="*60)
print("Clustering comparison summary")
print("="*60)
comparison_df = pd.DataFrame([
    {"method": "DBN + GMM", "n_clusters": N_CLUSTERS, "silhouette": dbn_sil, "ARI_vs_true_type": dbn_ari},
    {"method": "KMeans (raw)", "n_clusters": N_CLUSTERS, "silhouette": kmeans_sil, "ARI_vs_true_type": kmeans_ari},
])
print(comparison_df.to_string(index=False))


NMF flagged 98 windows | 13 distinct true labels present (incl. false positives labeled 'normal')
Using N_CLUSTERS = 12

1. DBN + GMM clustering
Latent shape after DBN: (98, 32)
Silhouette score (latent space): 0.5980
Adjusted Rand Index vs true anomaly_type: 0.0583

DBN+GMM cluster vs true anomaly_type:
dbn_cluster                       0   1   2   3   4   5   6   7   8   9   10  \
true_type                                                                      
appliance_unusual_hours            0   0   1   0   0   0   0   0   1   0   0   
gradual_drift_decrease             0   0   0   1   0   0   0   0   0   0   0   
gradual_drift_increase             0   0   0   0   1   0   0   0   0   0   0   
heating_on_warm_day                1   0   1   0   0   0   0   0   0   0   0   
impossible_appliance_combo         0   0   1   0   0   0   0   0   0   0   0   
multiple_high_power_simultaneous   1   0   1   0   0   0   0   0   0   0   0   
normal                            10   2   4   4   8 

Best Model Run - Reduced Dataset

In [1]:
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

# ============================================================
# BEST CONFIG — locked in from the alpha/K sweep
# ============================================================
BEST_ALPHA = 0.4
BEST_K = 40

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(r'pipeline_cache\ampds_behavior_context_reduced_features.csv')
df = df.drop(columns=['window_id'])

# ============================================================
# STRATIFIED SPLIT — same split used throughout, for consistency
# ============================================================
df_reset = df.reset_index(drop=True)
anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
normal_df = df_reset[df_reset['is_anomaly'] == 0]

anom_train, anom_temp = train_test_split(
    anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
)
anom_val, anom_test = train_test_split(
    anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
)
norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

drop_cols = ['is_anomaly', 'anomaly_type']
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_anomaly']
X_val = val_df.drop(columns=drop_cols)
y_val = val_df['is_anomaly']
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['is_anomaly']

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# ============================================================
# TRAIN ON NORMAL DATA ONLY, SCALE
# ============================================================
X_train_normal = X_train[y_train == 0]
print(f"Normal training rows: {len(X_train_normal)}")

scaler = MinMaxScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

# ============================================================
# SCORING FUNCTION — combined input-space + latent-space reconstruction error
# ============================================================
def rganomaly_score(nmf, X, alpha):
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)
    input_error = np.linalg.norm(X - X_recon, axis=1)

    W_reconstructed = nmf.transform(X_recon)
    latent_error = np.linalg.norm(W - W_reconstructed, axis=1)

    return alpha * input_error + (1 - alpha) * latent_error

# ============================================================
# FIT — raw, tight convergence, no efficiency shortcuts (single run, cost is trivial)
# ============================================================
print(f"\nFitting NMF: K={BEST_K}, alpha={BEST_ALPHA}")

start = time.perf_counter()
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    nmf = NMF(
        n_components=BEST_K,
        init="nndsvda",
        solver="cd",
        max_iter=3000,               
        random_state=42,
    )
    nmf.fit(X_train_normal_scaled)
    converged = len(w) == 0
fit_time = time.perf_counter() - start

print(f"Converged: {converged} | n_iter={nmf.n_iter_}/{nmf.max_iter} | fit_time={fit_time:.2f}s")

# ============================================================
# VALIDATION — threshold selection only
# ============================================================
val_scores = rganomaly_score(nmf, X_val_scaled, BEST_ALPHA)
val_roc_auc = roc_auc_score(y_val, val_scores)
val_pr_auc = average_precision_score(y_val, val_scores)

thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
for t in thresholds:
    preds = (val_scores >= t).astype(int)
    p = precision_score(y_val, preds, zero_division=0)
    r = recall_score(y_val, preds, zero_division=0)
    f1 = f1_score(y_val, preds, zero_division=0)
    if f1 > best["f1"]:
        best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

print(f"\nValidation: ROC-AUC={val_roc_auc:.4f} | PR-AUC={val_pr_auc:.4f} | "
      f"Best F1={best['f1']:.4f} @ threshold={best['threshold']:.6f}")

# ============================================================
# FINAL TEST EVALUATION
# ============================================================
test_scores = rganomaly_score(nmf, X_test_scaled, BEST_ALPHA)
test_pred = (test_scores >= best["threshold"]).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print("\n" + "="*60)
print(f"FINAL RESULT — NMF (K={BEST_K}, alpha={BEST_ALPHA})")
print("="*60)
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")
print(f"Confusion:  TP={tp}, FP={fp}, FN={fn}, TN={tn}")

# ============================================================
# PER-ANOMALY-TYPE BREAKDOWN
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_scores = rganomaly_score(nmf, X_sub_scaled, BEST_ALPHA)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print(per_type_df.to_string(index=False))

Train: (4098, 419) | Val: (879, 419) | Test: (879, 419)
Normal training rows: 3568

Fitting NMF: K=40, alpha=0.4
Converged: True | n_iter=1025/3000 | fit_time=9.06s

Validation: ROC-AUC=0.7631 | PR-AUC=0.3174 | Best F1=0.4629 @ threshold=0.739288

FINAL RESULT — NMF (K=40, alpha=0.4)
ROC AUC:    0.7399
PR AUC:     0.3203
Precision:  0.4592
Recall:     0.3947
F1:         0.4245
Confusion:  TP=45, FP=53, FN=69, TN=712

Per-anomaly-type AUC on test set:
                    anomaly_type      auc  n_anomaly
      weekday_pattern_on_weekend 0.978068          9
      weekend_pattern_on_weekday 0.976471          9
                   sensor_glitch 0.954902          4
      impossible_appliance_combo 0.952157          5
             stuck_appliance_off 0.943188         13
multiple_high_power_simultaneous 0.813333          5
             heating_on_warm_day 0.792157          9
          gradual_drift_decrease 0.691503          3
                     power_spike 0.686536          5
              s

Sweeping and finding best alpha and n_comps - Reduced Dataset

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import NMF
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix,
)

# ============================================================
# SWEEP CONFIG
# ============================================================
alphas = [0.2, 0.4, 0.5, 0.6, 0.8, 1.0]
n_components_list = [40, 60, 80, 100, 120, 150]

# ============================================================
# LOAD DATA
# ============================================================
df = pd.read_csv(r'pipeline_cache\ampds_behavior_context_reduced_features.csv')
df = df.drop(columns=['window_id'])

# ============================================================
# STRATIFIED SPLIT
# ============================================================
df_reset = df.reset_index(drop=True)
anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
normal_df = df_reset[df_reset['is_anomaly'] == 0]

anom_train, anom_temp = train_test_split(
    anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
)
anom_val, anom_test = train_test_split(
    anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
)
norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

drop_cols = ['is_anomaly', 'anomaly_type']
X_train = train_df.drop(columns=drop_cols)
y_train = train_df['is_anomaly']
X_val = val_df.drop(columns=drop_cols)
y_val = val_df['is_anomaly']
X_test = test_df.drop(columns=drop_cols)
y_test = test_df['is_anomaly']

print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

# ============================================================
# TRAIN ON NORMAL DATA ONLY, SCALE
# ============================================================
X_train_normal = X_train[y_train == 0]
print(f"Normal training rows: {len(X_train_normal)}")

scaler = MinMaxScaler()
X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

# ============================================================
# SCORING FUNCTION
# ============================================================
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.sum((X_ori - X_recon) ** 2, axis=1)

def rganomaly_score(nmf, X, alpha):
    W = nmf.transform(X)
    X_recon = nmf.inverse_transform(W)
    input_error = np.linalg.norm(X - X_recon, axis=1)

    W_reconstructed = nmf.transform(X_recon)
    latent_error = np.linalg.norm(W - W_reconstructed, axis=1)

    return alpha * input_error + (1 - alpha) * latent_error

def best_threshold_search(y_true, scores):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 99)))
    best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
    for t in thresholds:
        preds = (scores >= t).astype(int)
        p = precision_score(y_true, preds, zero_division=0)
        r = recall_score(y_true, preds, zero_division=0)
        f1 = f1_score(y_true, preds, zero_division=0)
        if f1 > best["f1"]:
            best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}
    return best

# ============================================================
# STEP 1 — FIT NMF ONCE PER K (alpha doesn't affect the fit, only the scoring)
# ============================================================
print("\n" + "="*60)
print("Fitting NMF for each K (alpha-independent step)")
print("="*60)

fitted_models = {}
for n_comps in n_components_list:
    start = time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        nmf = NMF(
            n_components=n_comps,
            init="nndsvda",
            solver="cd",
            max_iter=1500,
            tol=1e-3,
            random_state=42,
        )
        nmf.fit(X_train_normal_scaled)
        converged = len(w) == 0
    fit_time = time.perf_counter() - start

    train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
    train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

    fitted_models[n_comps] = {
        "model": nmf, "converged": converged, "n_iter": nmf.n_iter_,
        "fit_time": fit_time, "train_recon_err": train_err,
    }
    print(f"K={n_comps:>4} | {'OK' if converged else 'NOT CONVERGED':>13} | "
          f"n_iter={nmf.n_iter_:>5} | fit_time={fit_time:6.1f}s | train_err={train_err:.4f}")

# ============================================================
# STEP 2 — SCORE EACH (K, alpha) COMBO ON VALIDATION (cheap — no refit)
# ============================================================
print("\n" + "="*60)
print("Scoring alpha x K grid on validation")
print("="*60)

all_results = []
for n_comps, model_info in fitted_models.items():
    nmf = model_info["model"]
    for alpha in alphas:
        val_scores = rganomaly_score(nmf, X_val_scaled, alpha)
        roc_auc = roc_auc_score(y_val, val_scores)
        pr_auc = average_precision_score(y_val, val_scores)
        best = best_threshold_search(y_val, val_scores)

        all_results.append({
            "alpha": alpha, "n_components": n_comps,
            "converged": model_info["converged"], "n_iter": model_info["n_iter"],
            "fit_time_sec": round(model_info["fit_time"], 1),
            "train_recon_err": model_info["train_recon_err"],
            "roc_auc": roc_auc, "pr_auc": pr_auc,
            "best_val_f1": best["f1"], "best_val_precision": best["precision"],
            "best_val_recall": best["recall"], "best_threshold": best["threshold"],
        })

        print(f"K={n_comps:>4} alpha={alpha:.1f} | roc_auc={roc_auc:.4f} | pr_auc={pr_auc:.4f} | "
              f"best_f1={best['f1']:.4f}")

results_df = pd.DataFrame(all_results).sort_values(["pr_auc", "roc_auc"], ascending=False)

print("\n" + "="*80)
print("FULL SWEEP — Validation results (sorted by PR-AUC)")
print("="*80)
print(results_df.to_string(index=False))

# ============================================================
# STEP 3 — SELECT BEST (alpha, K) ON VALIDATION ONLY
# ============================================================
best_row = results_df.iloc[0]
best_alpha = float(best_row["alpha"])
best_k = int(best_row["n_components"])
best_model = fitted_models[best_k]["model"]
best_threshold = best_row["best_threshold"]

print(f"\nSelected on validation PR-AUC — alpha={best_alpha}, K={best_k}")
print(f"Validation threshold = {best_threshold:.6f}")
print(f"Converged: {fitted_models[best_k]['converged']} (n_iter={fitted_models[best_k]['n_iter']})")
print(f"Train recon err at this K: {fitted_models[best_k]['train_recon_err']:.4f} "
      f"(compare to neighboring K values above — flag if suspiciously low, sign of memorization)")

# ============================================================
# STEP 4 — TEST EVALUATION, TOUCHED EXACTLY ONCE
# ============================================================
test_scores = rganomaly_score(best_model, X_test_scaled, best_alpha)
test_pred = (test_scores >= best_threshold).astype(int)

test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

print("\n" + "="*60)
print(f"FINAL RESULT — NMF (K={best_k}, alpha={best_alpha}) — selected on val, tested once")
print("="*60)
print(f"ROC AUC:    {test_roc_auc:.4f}")
print(f"PR AUC:     {test_pr_auc:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1:         {test_f1:.4f}")
print(f"Confusion:  TP={tp}, FP={fp}, FN={fn}, TN={tn}")

# ============================================================
# STEP 5 — PER-ANOMALY-TYPE BREAKDOWN, best config only
# ============================================================
print("\nPer-anomaly-type AUC on test set:")
per_type_results = []
for atype in sorted(test_df['anomaly_type'].unique()):
    if atype == 'normal':
        continue
    mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
    sub = test_df[mask]
    X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
    sub_scores = rganomaly_score(best_model, X_sub_scaled, best_alpha)
    try:
        auc = roc_auc_score(sub['is_anomaly'], sub_scores)
    except ValueError:
        auc = float('nan')
    n_pos = (sub['is_anomaly'] == 1).sum()
    per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
print(per_type_df.to_string(index=False))

# ============================================================
# STEP 6 — top configs, for sanity-checking sensitivity around the winner
# ============================================================
print("\n" + "="*60)
print("Top 10 (alpha, K) configs by validation PR-AUC")
print("="*60)
print(results_df.head(10).to_string(index=False))